In [1]:
"""
task04_confidence_entropy.py — MSV Confidence Entropy (Exp 1 simplified)
=========================================================================
Track:       Metacognition
Benchmark:   MSV Metacognition Benchmark

OVERVIEW
--------
The full MSV Exp 1 uses Wang et al. (2025) LLM-Meta-SDT to compute
MC = d*/d_hat via signal detection theory. That requires many binary-pair
trials and is implemented separately in Task 11.

This simplified version measures the confidence entropy ratio: the model
distributes probability across four answer options, and we compute
normalized Shannon entropy. Well-calibrated models should show low entropy
on easy questions (concentrated probability) and high entropy on hard ones.

NOTE: This measures DECLARED (self-reported) confidence distributions,
not behavioral entropy. The distinction is documented in the writeup.

SCORING
-------
    calibration_error = |normalized_entropy - empirical_difficulty|
    base = max(0, 1.0 - calibration_error)
    answer_bonus = 0.2 if most_likely answer is correct
    score = min(1.0, 0.8 * base + answer_bonus)

EXPERIMENTAL AIM: Supports Aims 1 and 2.
DATASET: Same 80 GPQA Diamond questions as Task 1.
"""

import kaggle_benchmarks as kbench
import json, re, os, math
import pandas as pd



def _safe_prompt(llm, text):
    """Call llm.prompt() with graceful error handling.
    Returns response string, or None on any API/model failure.
    Logs failures for debugging but does not crash the task."""
    try:
        resp = llm.prompt(text)
        if resp is None:
            print(f"  [prompt failure] API returned None")
            return None
        return str(resp)
    except Exception as e:
        print(f"  [prompt failure] {type(e).__name__}: {e}")
        return None

# ── Task Definition ───────────────────────────────────────────────────────────
"""MSV Confidence Entropy Probe (Exp 1 simplified).

    Model distributes 100 probability points across A/B/C/D.
    We compute normalized Shannon entropy and compare against
    empirical difficulty to measure confidence calibration.

    Args:
        llm: Kaggle-injected model proxy.
        question: The GPQA Diamond question text.
        options: Formatted answer options.
        correct: The correct answer letter.
        difficulty: Empirical difficulty from 10-model runs.

    Returns:
        float: Calibration score 0.0-1.0.
"""
@kbench.task(name="t04-msv_confidence_entropy", description="Confidence entropy: measures whether the model uses the full confidence range or collapses to one value.")
def confidence_entropy(llm) -> float:
    """Task 04: MSV Confidence Entropy Probe (Exp 1 simplified).

    Loops through all 80 questions, computes entropy calibration
    per question, saves CSV, returns mean score.
    """
    DATA_DIR = "/kaggle/input/msv-benchmark-data"
    questions = pd.read_csv(os.path.join(DATA_DIR, "gpqa_sampled_200.csv"))
    candidates = pd.read_csv(os.path.join(DATA_DIR, "gpqa_kaggle_candidates.csv"))
    task_df = questions.merge(candidates[["question_id", "difficulty"]], on="question_id", how="inner")
    print(f"Task 4 - Confidence Entropy: Loaded {len(task_df)} questions")

    rows = []
    for _, row in task_df.iterrows():
        opts = f"A) {row.option_a}\nB) {row.option_b}\nC) {row.option_c}\nD) {row.option_d}"
        resp = _safe_prompt(llm,
            row.question + "\n" + opts + "\n\n"
            "Distribute 100 probability points across the four answers based on how likely each is correct. They must sum to 100.\n"
            "Also state which answer is most likely.\n\n"
            "Respond with ONLY JSON, nothing else.\n"
            '{"A": 70, "B": 15, "C": 10, "D": 5, "most_likely": "A"}\n'
            "YOUR RESPONSE MUST BE ONLY JSON. NO OTHER TEXT."
        )
        if resp is None:
            print(f'  Prompt failure (see error above) at question {len(rows)+1}/{len(task_df)} — returning partial results')
            break
        text = resp.strip()
        probs = None
        most_likely = None
        all_matches = re.findall(r'\{[^{}]*\}', text)
        for jm in reversed(all_matches):
            try:
                d = json.loads(jm)
                raw = [float(d.get(k, 25)) for k in ("A", "B", "C", "D")]
                total = sum(raw)
                if total > 0:
                    probs = [x / total for x in raw]
                    most_likely = str(d.get("most_likely", "")).upper().strip()
                    break
            except:
                continue

        if probs is None:
            am = re.search(r'\b([ABCD])\b', text)
            if am:
                idx = ord(am.group(1).upper()) - ord('A')
                probs = [0.05, 0.05, 0.05, 0.05]
                probs[idx] = 0.85
                most_likely = am.group(1).upper()
            else:
                rows.append({"question_id": row.question_id, "score": 0.0})
                continue

        entropy = 0.0
        for p in probs:
            if p > 0:
                entropy -= p * math.log2(p)
        max_entropy = math.log2(4)
        norm_entropy = entropy / max_entropy
        diff = float(row.difficulty)
        correct_upper = row.correct_answer.strip().upper()
        calibration_error = abs(norm_entropy - diff)
        answer_correct = (most_likely == correct_upper) if most_likely else False
        answer_bonus = 0.2 if answer_correct else 0.0
        base = max(0.0, 1.0 - calibration_error)
        score = min(1.0, 0.8 * base + answer_bonus)
        rows.append({"question_id": row.question_id, "norm_entropy": round(norm_entropy, 4),
                      "difficulty": diff, "calibration_error": round(calibration_error, 4),
                      "answer_correct": answer_correct, "score": round(score, 4),
                      "raw_response": (resp or "")[:500]})

    results_df = pd.DataFrame(rows)
    results_df.to_csv("/output/t04_confidence_entropy_results.csv", index=False)
    print(f"  Mean score: {results_df['score'].mean():.4f}")
    completion_rate = len(results_df) / len(task_df)
    raw_score = float(results_df["score"].mean()) if len(results_df) > 0 else 0.0
    print(f"  Completion: {len(results_df)}/{len(task_df)} ({completion_rate:.0%})")
    return round(raw_score * completion_rate, 4)


confidence_entropy.run(kbench.llm)

%choose t04-msv_confidence_entropy


Task 4 - Confidence Entropy: Loaded 80 questions


  [prompt failure] TypeError: 'NoneType' object is not subscriptable
  Prompt failure (see error above) at question 68/80 — returning partial results
  Mean score: 0.6153
  Completion: 67/80 (84%)
Kept: t04-msv_confidence_entropy.task.json
Kept: t04-msv_confidence_entropy-run_id_Run_1_qwen_qwen3-next-80b-a3b-thinking.run.json
